# FinBERT Fine-Tuning on MD&A Sentiment (CUDA)

**Pipeline**

1. Load `mda_shared_preprocessed.csv` from `sentiment_analysis_data/`
2. Time-based stratified split: filings ≤ 2022 → train, 2023+ → test
3. Fine-tune `ProsusAI/finbert` on CUDA GPU
4. Evaluate on held-out test set (macro F1 + classification report)


## 1. Imports & Device Setup


In [1]:
!pip install peft 

Defaulting to user installation because normal site-packages is not writeable


In [2]:
import warnings
import os

warnings.filterwarnings("ignore")
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
)
from datasets import Dataset as HFDataset
from sklearn.metrics import classification_report, f1_score

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

# ── Device ────────────────────────────────────────────────────────────────────
if not torch.cuda.is_available():
    raise RuntimeError("CUDA is not available. Check your GPU cluster environment.")
DEVICE = torch.device("cuda")

print(f"PyTorch : {torch.__version__}")
print(f"Device  : {DEVICE}  ({torch.cuda.get_device_name(0)})")


PyTorch : 2.10.0+cu128
Device  : cuda  (NVIDIA A40)


## 2. Load & Prepare Dataset


In [3]:
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

CSV_PATH = Path("~/sentiment_analysis_data/mda_shared_preprocessed.csv")
docs = pd.read_csv(CSV_PATH)
print(f"Loaded : {docs.shape[0]:,} rows from {CSV_PATH}")
docs.head()

Loaded : 452,390 rows from ~/sentiment_analysis_data/mda_shared_preprocessed.csv


,doc_id,company_name,filing_type,filing_date,year,quarter,sentence,sentiment
0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,presently the companys product line includes a...,neutral
1,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,approximately NUM of all sales were generated ...,neutral
2,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,on an ongoing basis we re-evaluate our judgmen...,neutral
3,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,we base our estimates and judgments on our his...,neutral
4,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,actual results may differ from these estimates...,neutral


In [4]:
print(f"Loaded : {docs.shape[0]:,} documents, {docs.shape[1]} columns")
print(f"Columns: {docs.columns.tolist()}")
print(f"Companies : {docs['company_name'].nunique():,}")
print(f"Year range: {docs['year'].min()} – {docs['year'].max()}")
print()
print("Filing type counts:")
print(docs["filing_type"].value_counts().to_string())


Loaded : 452,390 documents, 8 columns
Columns: ['doc_id', 'company_name', 'filing_type', 'filing_date', 'year', 'quarter', 'sentence', 'sentiment']
Companies : 473
Year range: 2010 – 2025

Filing type counts:
filing_type
10-K    237517
10-Q    214873


In [5]:
docs.head()

,doc_id,company_name,filing_type,filing_date,year,quarter,sentence,sentiment
0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,presently the companys product line includes a...,neutral
1,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,approximately NUM of all sales were generated ...,neutral
2,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,on an ongoing basis we re-evaluate our judgmen...,neutral
3,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,we base our estimates and judgments on our his...,neutral
4,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020-01-28,2020,Q1,actual results may differ from these estimates...,neutral


In [6]:
long_df = docs.rename(columns={"sentence": "text"}).copy()
long_df["text"] = long_df["text"].astype(str).str.strip()
long_df = long_df[long_df["text"].str.len() > 0].reset_index(drop=True)

MAX_SENTENCES = 500_000
if MAX_SENTENCES and len(long_df) > MAX_SENTENCES:
    long_df = (
        long_df.groupby("year", group_keys=False)
        .apply(lambda g: g.sample(frac=MAX_SENTENCES / len(long_df), random_state=SEED))
        .reset_index(drop=True)
    )
    print(f"Sampled down to {len(long_df):,} sentences (stratified by year)")

print(f"Documents : {docs.shape[0]:,}")
print(f"Sentences : {len(long_df):,}  (one row each)")
long_df[["doc_id", "company_name", "filing_type", "year", "quarter", "text"]].head(5)

Documents : 452,390
Sentences : 452,390  (one row each)


,doc_id,company_name,filing_type,year,quarter,text
0,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020,Q1,presently the companys product line includes a...
1,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020,Q1,approximately NUM of all sales were generated ...
2,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020,Q1,on an ongoing basis we re-evaluate our judgmen...
3,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020,Q1,we base our estimates and judgments on our his...
4,1-800-PetMeds_10-Q_2020-01-28,1-800-PetMeds,10-Q,2020,Q1,actual results may differ from these estimates...


## 3. FinBERT Architecture Configuration

FinBERT is built on **BERT-base** (`BertConfig`). Below are the four structural hyperparameters you can define — everything else is fixed by the pre-trained checkpoint.

| Parameter             | Default (FinBERT) | What it controls                                                                 |
| --------------------- | ----------------- | -------------------------------------------------------------------------------- |
| `num_hidden_layers`   | 12                | Number of Transformer encoder blocks stacked on top of each other                |
| `num_attention_heads` | 12                | Parallel attention "heads" per layer — each learns different token relationships |
| `hidden_size`         | 768               | Width of every hidden vector — **must be divisible by `num_attention_heads`**    |
| `intermediate_size`   | 3072              | Inner width of the feed-forward sub-layer (conventionally 4 × `hidden_size`)     |

**Total parameter count** scales roughly as:
`params ≈ num_layers × (4 × hidden² + 2 × hidden × intermediate)`

> **Important**: If you change `num_hidden_layers` / `hidden_size` / `num_attention_heads` away from the defaults, the pre-trained weights **cannot be loaded** for those layers — training starts from random. The values below match the original FinBERT so pre-trained weights are fully reused.


In [7]:
MODEL_NAME = "ProsusAI/finbert"

NUM_HIDDEN_LAYERS = 12
NUM_ATTENTION_HEADS = 12
HIDDEN_SIZE = 768
INTERMEDIATE_SIZE = 3072

LABEL2ID = {"positive": 0, "neutral": 1, "negative": 2}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
MAX_LEN = 128

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    num_hidden_layers=NUM_HIDDEN_LAYERS,
    num_attention_heads=NUM_ATTENTION_HEADS,
    hidden_size=HIDDEN_SIZE,
    intermediate_size=INTERMEDIATE_SIZE,
    ignore_mismatched_sizes=True,
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(
    f"Architecture : {NUM_HIDDEN_LAYERS} layers × {NUM_ATTENTION_HEADS} heads × {HIDDEN_SIZE}d hidden"
)
print(f"Total params : {total_params:,}")
print(f"Trainable    : {trainable:,}")
print(f"Device       : {next(model.parameters()).device}")


Architecture : 12 layers × 12 heads × 768d hidden
Total params : 109,484,547
Trainable    : 109,484,547
Device       : cuda:0


## 5. Time-Stratified Train / Test Split

Filings **≤ 2022** → train, **2023+** → test.  
This prevents look-ahead bias — the model is evaluated only on filings filed after the training period.


In [8]:
import os
from pathlib import Path

TRAIN_CUTOFF = 2022  # filings up to and including 2022 → train; 2023+ → test

# ── Label encoding (LABEL2ID defined in § 3 architecture cell) ────────────────
long_df["label"] = long_df["sentiment"].str.strip().str.lower()
long_df = long_df[long_df["label"].isin(LABEL2ID)].reset_index(drop=True)
long_df["label_id"] = long_df["label"].map(LABEL2ID)

train_df = long_df[long_df["year"] <= TRAIN_CUTOFF].copy().reset_index(drop=True)
test_df = long_df[long_df["year"] > TRAIN_CUTOFF].copy().reset_index(drop=True)

print(f"Train (≤ {TRAIN_CUTOFF}) : {len(train_df):>8,} sentences")
print(f"Test  ({TRAIN_CUTOFF + 1}+)  : {len(test_df):>8,} sentences")
print()
print("Train label distribution:")
print(train_df["label"].value_counts().to_string())
print()
print("Test label distribution:")
print(test_df["label"].value_counts().to_string())


# ── Tokenise ──────────────────────────────────────────────────────────────────
# num_proc=1: HuggingFace fast tokenizer is already parallelised in Rust;
# spawning multiple processes inside a notebook adds IPC overhead with no gain.
# writer_batch_size=10_000 flushes Arrow batches less often → faster disk writes.
def tokenize(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LEN)


N_PROC = 1  # fast tokenizer is multi-threaded internally

CACHE_DIR = Path("./hf_cache")
TRAIN_CACHE = CACHE_DIR / "train"
TEST_CACHE = CACHE_DIR / "test"

if TRAIN_CACHE.exists() and TEST_CACHE.exists():
    from datasets import load_from_disk

    hf_train = load_from_disk(str(TRAIN_CACHE))
    hf_test = load_from_disk(str(TEST_CACHE))
    print("Loaded tokenised datasets from cache.")
else:
    hf_train = HFDataset.from_dict(
        {"text": train_df["text"].tolist(), "labels": train_df["label_id"].tolist()}
    )
    hf_test = HFDataset.from_dict(
        {"text": test_df["text"].tolist(), "labels": test_df["label_id"].tolist()}
    )
    hf_train = hf_train.map(
        tokenize,
        batched=True,
        batch_size=10_000,
        num_proc=N_PROC,
        remove_columns=["text"],
    )
    hf_test = hf_test.map(
        tokenize,
        batched=True,
        batch_size=10_000,
        num_proc=N_PROC,
        remove_columns=["text"],
    )
    hf_train.save_to_disk(str(TRAIN_CACHE))
    hf_test.save_to_disk(str(TEST_CACHE))
    print("Tokenised and cached to ./hf_cache/")

print(f"\nHF train size : {len(hf_train):,}")
print(f"HF test  size : {len(hf_test):,}")


Train (≤ 2022) :  341,359 sentences
Test  (2023+)  :  111,031 sentences

Train label distribution:
label
neutral     284435
positive     29097
negative     27827

Test label distribution:
label
neutral     93951
negative     8613
positive     8467
Loaded tokenised datasets from cache.

HF train size : 194,876
HF test  size : 58,979


## 6. Fine-Tune FinBERT


In [9]:
import numpy as np
from sklearn.metrics import f1_score


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro")}


model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=3,
    id2label=ID2LABEL,
    label2id=LABEL2ID,
    num_hidden_layers=NUM_HIDDEN_LAYERS,
    num_attention_heads=NUM_ATTENTION_HEADS,
    hidden_size=HIDDEN_SIZE,
    intermediate_size=INTERMEDIATE_SIZE,
    ignore_mismatched_sizes=True,
).to(DEVICE)

training_args = TrainingArguments(
    output_dir="./finbert_mda_checkpoints",
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=256,
    num_train_epochs=3,
    fp16=True,
    bf16=False,
    group_by_length=True,
    optim="adamw_torch_fused",
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    eval_accumulation_steps=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    logging_steps=200,
    seed=SEED,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=hf_train,
    eval_dataset=hf_test,
    tokenizer=tokenizer,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
)

print("Starting fine-tuning...")
trainer.train()


Starting fine-tuning...


Epoch,Training Loss,Validation Loss,Macro F1
1,0.102300,0.134072,0.889677
2,0.078300,0.126180,0.902490
3,0.031200,0.154407,0.909063


TrainOutput(global_step=18270, training_loss=0.10795572662718908, metrics={'train_runtime': 866.34, 'train_samples_per_second': 674.825, 'train_steps_per_second': 21.089, 'total_flos': 1.0885322760885216e+16, 'train_loss': 0.10795572662718908, 'epoch': 3.0})

## 8. Evaluation on Test Set


In [10]:
preds_out = trainer.predict(hf_test)
y_pred = np.argmax(preds_out.predictions, axis=-1)
y_true = preds_out.label_ids

pred_labels = [ID2LABEL[p] for p in y_pred]
true_labels = [ID2LABEL[t] for t in y_true]

print("=== Fine-tuned FinBERT on MD&A — Test Set (2023+) ===")
print(
    classification_report(
        true_labels,
        pred_labels,
        target_names=["positive", "neutral", "negative"],
        zero_division=0,
    )
)

macro_f1 = f1_score(y_true, y_pred, average="macro")
print(f"Macro F1: {macro_f1:.4f}")


=== Fine-tuned FinBERT on MD&A — Test Set (2023+) ===
              precision    recall  f1-score   support

    positive       0.88      0.88      0.88      4575
     neutral       0.98      0.98      0.98     49743
    negative       0.87      0.87      0.87      4661

    accuracy                           0.96     58979
   macro avg       0.91      0.91      0.91     58979
weighted avg       0.96      0.96      0.96     58979

Macro F1: 0.9091
